# U.S. Motor Bus Ridership Data Preparation and Panel Construction
This notebook prepares the monthly U.S. Motor Bus ridership dataset
from raw NTD files into a clean State–Year–Month panel format.
The output dataset is used as the dependent variable input for
structural break analysis in the Capstone Project.

In [78]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import prettytable

prettytable.DEFAULT = 'DEFAULT'

current = Path().resolve()

while current != current.parent:
    if (current / "Data").exists():
        PROJECT_ROOT = current
        break
    current = current.parent
else:
    raise FileNotFoundError("Project root with 'Data' folder not found")

DATA_PROCESSED = PROJECT_ROOT / "Data" / "Processed"
DATA_CLEAN = PROJECT_ROOT / "Data" / "Cleaned"

DATA_CLEAN.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Processed dir:", DATA_PROCESSED)
print("Clean dir:", DATA_CLEAN)


Project root: /Users/emerino/Documents/Capstone_Project_Group_4-
Processed dir: /Users/emerino/Documents/Capstone_Project_Group_4-/Data/Processed
Clean dir: /Users/emerino/Documents/Capstone_Project_Group_4-/Data/Cleaned


## 1) Load the Excel file

In [79]:
DATA_VERSION = "complete_fta_dec2025"

# Point this to your COMPLETE file (Windows example):
#excel_path = Path(r"C:\Users\djsj0\OneDrive\Documents\GitHub\Capstone_Project_Group_4-\Data\Processed\December 2025 Complete Monthly Ridership (with adjustments and estimates)_260202.xlsx")
excel_path = DATA_PROCESSED / "December 2025 Complete Monthly Ridership (with adjustments and estimates)_260202.xlsx"

# Output filenames (so you don't overwrite RAW outputs)
OUT_PANEL = f"ridership_state_month_panel_MB_{DATA_VERSION}.csv"
OUT_FARES = f"fares_state_fy_MB_{DATA_VERSION}.csv"

print("Using file:", excel_path)
print("Exists:", excel_path.exists())

Using file: /Users/emerino/Documents/Capstone_Project_Group_4-/Data/Processed/December 2025 Complete Monthly Ridership (with adjustments and estimates)_260202.xlsx
Exists: True


## 2) Data Inspection and Column Standardization

In [80]:
# Load the two relevant sheets:
# - UPT: monthly unlinked passenger trips in wide format
# - Master: agency metadata including HQ State (for state mapping)
upt = pd.read_excel(excel_path, sheet_name="UPT")
master = pd.read_excel(excel_path, sheet_name="Master")

# Normalize column names (strip spaces)
upt.columns = [str(c).strip() for c in upt.columns]
master.columns = [str(c).strip() for c in master.columns]

print("UPT shape:", upt.shape)
print("Master shape:", master.shape)
print("UPT columns (first 15):", upt.columns.tolist()[:15])
print("Master columns:", master.columns.tolist())

UPT shape: (2352, 298)
Master shape: (2340, 27)
UPT columns (first 15): ['NTD ID', 'Legacy NTD ID', 'Agency', 'Mode/Type of Service Status', 'Reporter Type', 'UACE CD', 'UZA Name', 'Mode', 'TOS', '3 Mode', '1/2002', '2/2002', '3/2002', '4/2002', '5/2002']
Master columns: ['NTD ID', 'Legacy NTD ID', 'Agency', 'Mode', 'TOS', '3 Mode', 'Mode/Type of Service Status', 'Reporter Type', 'Organization Type', 'HQ City', 'HQ State', 'UACE CD', 'UZA Name', 'UZA SQ Miles', 'UZA Population', 'Service Area Population', 'Service Area SQ Miles', 'Last Closed Report Year', 'Last Closed FY End Month', 'Last Closed FY End Year', 'Passenger Miles FY', 'Unlinked Passenger Trips FY', 'Avg Trip Length FY', 'Fares FY', 'Operating Expenses FY', 'Avg Cost Per Trip FY', 'Avg Fares Per Trip FY']


## 3) Identify monthly columns in UPT (e.g., `1/2002`, `12/2025`)

In [81]:
# Monthly columns look like M/YYYY (1 or 2 digits, slash, 4-digit year)
month_col_pattern = re.compile(r"^\d{1,2}/\d{4}$")

id_cols = [c for c in upt.columns if not month_col_pattern.match(c)]
month_cols = [c for c in upt.columns if month_col_pattern.match(c)]

print("ID columns detected:", id_cols[:15], "...")
print("Monthly columns detected:", len(month_cols))
print("Monthly columns sample:", month_cols[:10])

ID columns detected: ['NTD ID', 'Legacy NTD ID', 'Agency', 'Mode/Type of Service Status', 'Reporter Type', 'UACE CD', 'UZA Name', 'Mode', 'TOS', '3 Mode'] ...
Monthly columns detected: 288
Monthly columns sample: ['1/2002', '2/2002', '3/2002', '4/2002', '5/2002', '6/2002', '7/2002', '8/2002', '9/2002', '10/2002']


In [82]:
# -----------------------------
# FY Fares extraction (Motor Bus) from Master
# -----------------------------

# Normalize strings (safe)
for c in ["Agency", "Mode", "TOS"]:
    if c in master.columns:
        master[c] = master[c].astype(str).str.strip()

# Keep only Motor Bus
master_mb = master[master["Mode"].astype(str).str.strip().eq("MB")].copy()

# Columns we will use (only keep those that exist)
needed = [
    "NTD ID", "Agency", "Mode", "TOS", "HQ State",
    "Last Closed FY End Year", "Last Closed FY End Month",
    "Unlinked Passenger Trips FY",
    "Fares FY",
    "Operating Expenses FY",
    "Avg Fares Per Trip FY",
    "Avg Cost Per Trip FY"
]
needed = [c for c in needed if c in master_mb.columns]
master_mb = master_mb[needed].copy()

# Coerce numeric
num_cols = [c for c in [
    "Last Closed FY End Year", "Last Closed FY End Month",
    "Unlinked Passenger Trips FY", "Fares FY", "Operating Expenses FY",
    "Avg Fares Per Trip FY", "Avg Cost Per Trip FY"
] if c in master_mb.columns]

for c in num_cols:
    master_mb[c] = pd.to_numeric(master_mb[c], errors="coerce")

# Rename to consistent naming
master_mb = master_mb.rename(columns={
    "HQ State": "state",
    "Last Closed FY End Year": "FY_End_Year",
    "Last Closed FY End Month": "FY_End_Month",
    "Unlinked Passenger Trips FY": "UPT_FY",
    "Fares FY": "Fares_FY",
    "Operating Expenses FY": "OperatingExpenses_FY",
    "Avg Fares Per Trip FY": "AvgFaresPerTrip_FY",
    "Avg Cost Per Trip FY": "AvgCostPerTrip_FY"
})

# Drop rows missing essentials for aggregation
master_mb = master_mb.dropna(subset=["state", "FY_End_Year"])

# Build State–FY table (summing across agencies)
fares_state_fy = (
    master_mb
    .groupby(["state", "FY_End_Year"], as_index=False)
    .agg(
        Fares_FY_Total=("Fares_FY", "sum"),
        OperatingExpenses_FY_Total=("OperatingExpenses_FY", "sum"),
        UPT_FY_Total=("UPT_FY", "sum")
    )
)

# Derived ratios (more defensible than averaging "AvgFaresPerTrip_FY")
fares_state_fy["FarePerTrip_FY_Derived"] = (
    fares_state_fy["Fares_FY_Total"] / fares_state_fy["UPT_FY_Total"]
)

fares_state_fy["FareboxRecovery_FY_Derived"] = (
    fares_state_fy["Fares_FY_Total"] / fares_state_fy["OperatingExpenses_FY_Total"]
)

# Export
#fares_output_path = CLEAN_DIR / OUT_FARES
fares_output_path = DATA_CLEAN / OUT_FARES

fares_state_fy.to_csv(fares_output_path, index=False)

print("Saved to:", fares_output_path)
fares_state_fy.head()

Saved to: /Users/emerino/Documents/Capstone_Project_Group_4-/Data/Cleaned/fares_state_fy_MB_complete_fta_dec2025.csv


,state,FY_End_Year,Fares_FY_Total,OperatingExpenses_FY_Total,UPT_FY_Total,FarePerTrip_FY_Derived,FareboxRecovery_FY_Derived
0,AK,2019.0,233684.0,2115869.0,158949.0,1.470182,0.110444
1,AK,2024.0,3232273.0,33876425.0,3044346.0,1.061730,0.095414
2,AL,2003.0,2000117.0,13568610.0,3673280.0,0.544504,0.147408
3,AL,2024.0,2102070.0,42168937.0,3188514.0,0.659263,0.049849
4,AR,2024.0,1233613.0,15912523.0,1746467.0,0.706348,0.077525


## 4) Reshape to long format (one row per Agency–Mode–TOS–Month)
We convert from wide monthly columns into a tidy panel format.

In [83]:
upt_long = upt.melt(
    id_vars=id_cols,
    value_vars=month_cols,
    var_name="MonthYear",
    value_name="UPT"
)

# Parse Month and Year from strings like "3/2020"
upt_long["Month"] = upt_long["MonthYear"].str.split("/").str[0].astype(int)
upt_long["Year"] = upt_long["MonthYear"].str.split("/").str[1].astype(int)

# Clean ridership values
upt_long["UPT"] = pd.to_numeric(upt_long["UPT"], errors="coerce")

# -----------------------------
# Validation: Check numeric coercion impact
# -----------------------------

# Count original missing values BEFORE coercion
original_missing = upt_long["UPT"].isna().sum()

# Recreate a copy BEFORE coercion (if you didn't save one earlier)
# If you want to be more rigorous, create a temporary column:
upt_long["_UPT_original"] = upt_long["UPT"]

# Now re-run coercion cleanly for validation
upt_long["UPT"] = pd.to_numeric(upt_long["_UPT_original"], errors="coerce")

after_missing = upt_long["UPT"].isna().sum()

print("Missing BEFORE coercion:", original_missing)
print("Missing AFTER coercion:", after_missing)
print("New NaNs introduced by coercion:", after_missing - original_missing)

# Inspect suspicious rows (if any)
if after_missing > original_missing:
    print("Rows where coercion introduced NaN:")
    display(
        upt_long.loc[
            upt_long["_UPT_original"].notna() & upt_long["UPT"].isna(),
            ["Agency", "Mode", "TOS", "MonthYear", "_UPT_original"]
        ].head(20)
    )
else:
    print("No valid numeric values were lost during coercion.")

# Drop rows with no ridership
upt_long = upt_long.dropna(subset=["UPT"])

print("Long format shape:", upt_long.shape)
upt_long.head()

Missing BEFORE coercion: 312285
Missing AFTER coercion: 312285
New NaNs introduced by coercion: 0
No valid numeric values were lost during coercion.
Long format shape: (365091, 15)


,NTD ID,Legacy NTD ID,Agency,Mode/Type of Service Status,Reporter Type,UACE CD,UZA Name,Mode,TOS,3 Mode,MonthYear,UPT,Month,Year,_UPT_original
0,1.0,0001,King County,Active,Full Reporter,80389.0,"Seattle--Tacoma, WA",DR,PT,Bus,1/2002,135144.0,1,2002,135144.0
3,1.0,0001,King County,Inactive,Full Reporter,80389.0,"Seattle--Tacoma, WA",LR,DO,Rail,1/2002,12990.0,1,2002,12990.0
4,1.0,0001,King County,Active,Full Reporter,80389.0,"Seattle--Tacoma, WA",MB,DO,Bus,1/2002,6045861.0,1,2002,6045861.0
7,1.0,0001,King County,Active,Full Reporter,80389.0,"Seattle--Tacoma, WA",TB,DO,Bus,1/2002,2044815.0,1,2002,2044815.0
8,1.0,0001,King County,Active,Full Reporter,80389.0,"Seattle--Tacoma, WA",VP,DO,Bus,1/2002,174040.0,1,2002,174040.0


## 5) Filter to Motor Bus only (Mode = MB)
Your Capstone focuses on public bus ridership, so we keep MB only.

In [84]:
# Ensure Mode exists
if "Mode" not in upt_long.columns:
    raise KeyError("Column 'Mode' not found in UPT sheet after melting. Check the sheet structure.")

upt_mb = upt_long[upt_long["Mode"].astype(str).str.strip().eq("MB")].copy()

print("Motor Bus-only shape:", upt_mb.shape)
upt_mb[["Agency","Mode","TOS","Year","Month","UPT"]].head()

Motor Bus-only shape: (141545, 15)


,Agency,Mode,TOS,Year,Month,UPT
4,King County,MB,DO,2002,1,6045861.0
11,Spokane Transit Authority,MB,DO,2002,1,749011.0
15,Pierce County Transportation Benefit Area Auth...,MB,DO,2002,1,1102076.0
18,City of Everett,MB,DO,2002,1,131902.0
21,City of Yakima,MB,DO,2002,1,0.0


## 6) Add State: merge HQ State from Master

UPT sheet does not contain a clean State column for your state-month panel.
We merge `HQ State` from the `Master` sheet using common keys.

Keys used: `NTD ID`, `Agency`, `Mode`, `TOS`

## State Identification Method

The UPT (Unlinked Passenger Trips) sheet does not contain a direct "State" column.

To construct a state-level panel dataset, we use the "HQ State" variable from the Master sheet and merge it using the keys:
- NTD ID
- Agency
- Mode
- TOS

This approach assigns each agency's ridership activity to the state in which its headquarters (HQ State) is located.

While this method assumes that agency activity is primarily concentrated within its HQ state, it provides a consistent and reproducible mechanism for constructing a State–Year–Month panel suitable for structural break analysis.

In [85]:
# Check required columns exist for merge
for col in ["NTD ID", "Agency", "Mode", "TOS"]:
    if col not in upt_mb.columns:
        raise KeyError(f"UPT data missing merge key column: {col}")
for col in ["NTD ID", "Agency", "Mode", "TOS", "HQ State"]:
    if col not in master.columns:
        # Master has HQ State in your file; if not found, list columns for debugging.
        raise KeyError(f"Master sheet missing required column: {col}. Available: {master.columns.tolist()}")

# Create a compact mapping table
state_map = master[["NTD ID", "Agency", "Mode", "TOS", "HQ State"]].copy()
state_map = state_map.rename(columns={"HQ State": "State"})

# ----------------------------------
# Convert state abbreviations to full names
# ----------------------------------

state_name_map = {
    "AL": "Alabama",
    "AK": "Alaska",
    "AZ": "Arizona",
    "AR": "Arkansas",
    "CA": "California",
    "CO": "Colorado",
    "CT": "Connecticut",
    "DE": "Delaware",
    "FL": "Florida",
    "GA": "Georgia",
    "HI": "Hawaii",
    "ID": "Idaho",
    "IL": "Illinois",
    "IN": "Indiana",
    "IA": "Iowa",
    "KS": "Kansas",
    "KY": "Kentucky",
    "LA": "Louisiana",
    "ME": "Maine",
    "MD": "Maryland",
    "MA": "Massachusetts",
    "MI": "Michigan",
    "MN": "Minnesota",
    "MS": "Mississippi",
    "MO": "Missouri",
    "MT": "Montana",
    "NE": "Nebraska",
    "NV": "Nevada",
    "NH": "New Hampshire",
    "NJ": "New Jersey",
    "NM": "New Mexico",
    "NY": "New York",
    "NC": "North Carolina",
    "ND": "North Dakota",
    "OH": "Ohio",
    "OK": "Oklahoma",
    "OR": "Oregon",
    "PA": "Pennsylvania",
    "RI": "Rhode Island",
    "SC": "South Carolina",
    "SD": "South Dakota",
    "TN": "Tennessee",
    "TX": "Texas",
    "UT": "Utah",
    "VT": "Vermont",
    "VA": "Virginia",
    "WA": "Washington",
    "WV": "West Virginia",
    "WI": "Wisconsin",
    "WY": "Wyoming",
    "DC": "District of Columbia",
    "PR": "Puerto Rico",
    "VI": "Virgin Islands",
    "GU": "Guam"
}

state_map["State"] = state_map["State"].map(state_name_map)

# Normalize string columns for safer merging
for c in ["Agency", "Mode", "TOS"]:
    state_map[c] = state_map[c].astype(str).str.strip()
    upt_mb[c] = upt_mb[c].astype(str).str.strip()

# Merge
upt_mb = upt_mb.merge(
    state_map,
    on=["NTD ID", "Agency", "Mode", "TOS"],
    how="left",
    validate="m:1"  # many months -> one agency/mode/tos record in master
)

missing_state = upt_mb["State"].isna().mean()
print(f"Share of rows with missing State after merge: {missing_state:.2%}")

# Optional: if a small number are missing, you can inspect them:
# upt_mb[upt_mb["HQ State"].isna()][["NTD ID","Agency","Mode","TOS"]].drop_duplicates().head(20)

Share of rows with missing State after merge: 0.00%


## 7) Build the State–Year–Month panel

We aggregate Motor Bus ridership to the state-month unit of analysis.

In [86]:
# Ensure we have a consistent State column name
if "HQ State" in upt_mb.columns and "State" not in upt_mb.columns:
    upt_mb = upt_mb.rename(columns={"HQ State": "State"})

# Drop rows with missing state
panel_source = upt_mb.dropna(subset=["State"]).copy()

panel_state_month = (
    panel_source
    .groupby(["State", "Year", "Month"], as_index=False)
    .agg(Ridership_UPT=("UPT", "sum"))
    .sort_values(["State", "Year", "Month"])
)

# Post-COVID indicator for structural break modeling
panel_state_month["PostCOVID"] = (
    (panel_state_month["Year"] > 2020) |
    ((panel_state_month["Year"] == 2020) & (panel_state_month["Month"] >= 4))
).astype(int)

print("State–Year–Month panel shape:", panel_state_month.shape)
panel_state_month.head()

# ----------------------------------
# Validate Missing State-Month Combinations
# ----------------------------------

# Get full time range from your dataset
min_year = panel_state_month["Year"].min()
max_year = panel_state_month["Year"].max()

print("Time range:", min_year, "to", max_year)

# ==========================================================
# Balance check for 2015–2025 only (keep full 2002–2025 data)
# Allow up to 3 missing months per state
# ==========================================================

CHECK_START_YEAR = 2015
CHECK_END_YEAR   = 2025
MAX_MISSING_MONTHS_ALLOWED = 3

# Ensure merge keys are consistent types
panel_state_month["Year"] = panel_state_month["Year"].astype(int)
panel_state_month["Month"] = panel_state_month["Month"].astype(int)
panel_state_month["State"] = panel_state_month["State"].astype(str).str.strip()

# Build expected State-Year-Month grid ONLY for the check period
full_time_check = (
    pd.MultiIndex.from_product(
        [
            sorted(panel_state_month["State"].unique()),
            range(CHECK_START_YEAR, CHECK_END_YEAR + 1),
            range(1, 13)
        ],
        names=["State", "Year", "Month"]
    )
    .to_frame(index=False)
)

# Compare ONLY against observed rows in the check period
observed_check = panel_state_month[
    (panel_state_month["Year"] >= CHECK_START_YEAR) &
    (panel_state_month["Year"] <= CHECK_END_YEAR)
][["State", "Year", "Month"]].drop_duplicates()

merged_check = full_time_check.merge(
    observed_check,
    on=["State", "Year", "Month"],
    how="left",
    indicator=True
)

missing_state_month = merged_check[merged_check["_merge"] == "left_only"][["State", "Year", "Month"]]

print("Total missing state-month combinations (2015–2025 check):", len(missing_state_month))

missing_summary = (
    missing_state_month
    .groupby("State")
    .size()
    .sort_values(ascending=False)
)

print("States with missing months (2015–2025):")
print(missing_summary.head(20))

# Remove only states with more than allowed missing months
states_with_gaps = missing_summary[missing_summary > MAX_MISSING_MONTHS_ALLOWED].index.tolist()

print("States removed (missing months >", MAX_MISSING_MONTHS_ALLOWED, "):", states_with_gaps)

print("Original panel shape:", panel_state_month.shape)

balanced_panel = panel_state_month[
    ~panel_state_month["State"].isin(states_with_gaps)
].copy()

print("Balanced panel shape:", balanced_panel.shape)

# Replace main panel with balanced version
panel_state_month = balanced_panel.copy()

State–Year–Month panel shape: (14805, 5)
Time range: 2002 to 2025
Total missing state-month combinations (2015–2025 check): 276
States with missing months (2015–2025):
State
Virgin Islands    132
Wyoming           132
Idaho              12
dtype: int64
States removed (missing months > 3 ): ['Virgin Islands', 'Wyoming', 'Idaho']
Original panel shape: (14805, 5)
Balanced panel shape: (14376, 5)


In [87]:
# -------------------------------------------------
# Remove non-state geographies
# -------------------------------------------------

exclude = ["Puerto Rico", "District of Columbia"]

panel_state_month = panel_state_month[
    ~panel_state_month["State"].isin(exclude)
].copy()

print("Excluded:", exclude)

Excluded: ['Puerto Rico', 'District of Columbia']


In [88]:
# Number of unique states in final panel
num_states = panel_state_month["State"].nunique()

print("Number of states in final ridership panel:", num_states)

# List them sorted
states_list = sorted(panel_state_month["State"].unique())

print("\nStates included:")
for s in states_list:
    print(s)

Number of states in final ridership panel: 48

States included:
Alabama
Alaska
Arizona
Arkansas
California
Colorado
Connecticut
Delaware
Florida
Georgia
Hawaii
Illinois
Indiana
Iowa
Kansas
Kentucky
Louisiana
Maine
Maryland
Massachusetts
Michigan
Minnesota
Mississippi
Missouri
Montana
Nebraska
Nevada
New Hampshire
New Jersey
New Mexico
New York
North Carolina
North Dakota
Ohio
Oklahoma
Oregon
Pennsylvania
Rhode Island
South Carolina
South Dakota
Tennessee
Texas
Utah
Vermont
Virginia
Washington
West Virginia
Wisconsin


In [89]:
official_states = [
    "Alabama","Alaska","Arizona","Arkansas","California","Colorado",
    "Connecticut","Delaware","Florida","Georgia","Hawaii","Idaho",
    "Illinois","Indiana","Iowa","Kansas","Kentucky","Louisiana",
    "Maine","Maryland","Massachusetts","Michigan","Minnesota",
    "Mississippi","Missouri","Montana","Nebraska","Nevada",
    "New Hampshire","New Jersey","New Mexico","New York",
    "North Carolina","North Dakota","Ohio","Oklahoma","Oregon",
    "Pennsylvania","Rhode Island","South Carolina","South Dakota",
    "Tennessee","Texas","Utah","Vermont","Virginia","Washington",
    "West Virginia","Wisconsin","Wyoming"
]

In [90]:
current_states = sorted(panel_state_month["State"].unique())

missing_states = sorted(set(official_states) - set(current_states))

print("Missing states:", missing_states)
print("Count missing:", len(missing_states))

Missing states: ['Idaho', 'Wyoming']
Count missing: 2


## 8) VRM extraction (Motor Bus) and merge into the state-month panel

In [91]:
# 1) Load VRM sheet
vrm = pd.read_excel(excel_path, sheet_name="VRM")
vrm.columns = [str(c).strip() for c in vrm.columns]

# 2) Detect month columns (same pattern: M/YYYY)
month_col_pattern = re.compile(r"^\d{1,2}/\d{4}$")
vrm_id_cols = [c for c in vrm.columns if not month_col_pattern.match(str(c))]
vrm_month_cols = [c for c in vrm.columns if month_col_pattern.match(str(c))]

# 3) Wide -> long
vrm_long = vrm.melt(
    id_vars=vrm_id_cols,
    value_vars=vrm_month_cols,
    var_name="MonthYear",
    value_name="VRM"
)

# 4) Parse Month/Year + numeric conversion
vrm_long["Month"] = vrm_long["MonthYear"].str.split("/").str[0].astype(int)
vrm_long["Year"] = vrm_long["MonthYear"].str.split("/").str[1].astype(int)
vrm_long["VRM"] = pd.to_numeric(vrm_long["VRM"], errors="coerce")
vrm_long = vrm_long.dropna(subset=["VRM"])
vrm_long = vrm_long[vrm_long["Month"].between(1, 12)]

# 5) Filter to Motor Bus
vrm_long["Mode"] = vrm_long["Mode"].astype(str).str.strip()
vrm_long["TOS"] = vrm_long["TOS"].astype(str).str.strip()
vrm_long["Agency"] = vrm_long["Agency"].astype(str).str.strip()

vrm_mb = vrm_long[vrm_long["Mode"].eq("MB")].copy()

# 6) Merge State (same approach using HQ State from Master)
# (state_map should already exist from your UPT pipeline; if not, create it as below)
# state_map = master[["NTD ID", "Agency", "Mode", "TOS", "HQ State"]].rename(columns={"HQ State":"State"}).copy()

for c in ["Agency", "Mode", "TOS"]:
    vrm_mb[c] = vrm_mb[c].astype(str).str.strip()

vrm_mb = vrm_mb.merge(
    state_map,
    on=["NTD ID", "Agency", "Mode", "TOS"],
    how="left",
    validate="m:1"
)

vrm_mb = vrm_mb.dropna(subset=["State"])

# 7) Aggregate to State–Year–Month
vrm_state_month = (
    vrm_mb
    .groupby(["State", "Year", "Month"], as_index=False)
    .agg(VRM=("VRM", "sum"))
    .sort_values(["State", "Year", "Month"])
)

# 8) Merge VRM into your existing panel_state_month
# IMPORTANT: rename your ridership column to UPT for clarity before merging (recommended)
panel_state_month = panel_state_month.rename(columns={"Ridership_UPT": "UPT"})

panel_state_month = panel_state_month.merge(
    vrm_state_month,
    on=["State", "Year", "Month"],
    how="left",
    validate="1:1"
)

# Optional: if VRM should never be missing in the Complete dataset, you can check:
print("Share of missing VRM after merge:", panel_state_month["VRM"].isna().mean())

Share of missing VRM after merge: 0.0017391304347826088


In [92]:
# ----------------------------------
# Validate completeness for BOTH UPT and VRM
# ----------------------------------

# 1) Identify missing VRM rows (state-month exists but VRM is NaN)
missing_vrm_rows = panel_state_month[panel_state_month["VRM"].isna()].copy()
print("State-month rows with missing VRM:", len(missing_vrm_rows))

# 2) Count missing VRM months by state
missing_vrm_by_state = (
    missing_vrm_rows
    .groupby("State")
    .size()
    .sort_values(ascending=False)
)

print("States with missing VRM months (top):")
print(missing_vrm_by_state.head(20))

# ----------------------------------
# VRM completeness filter (2015–2025 only)
# Allow <= 3 missing months per state
# Keep full 2002–2025 data
# ----------------------------------

CHECK_START_YEAR = 2015
CHECK_END_YEAR   = 2025
MAX_MISSING_MONTHS_ALLOWED = 3

check_df = panel_state_month[
    (panel_state_month["Year"] >= CHECK_START_YEAR) &
    (panel_state_month["Year"] <= CHECK_END_YEAR)
].copy()

missing_vrm_rows = check_df[check_df["VRM"].isna()].copy()

missing_vrm_by_state = (
    missing_vrm_rows
    .groupby("State")
    .size()
    .sort_values(ascending=False)
)

states_with_vrm_gaps = missing_vrm_by_state[
    missing_vrm_by_state > MAX_MISSING_MONTHS_ALLOWED
].index.tolist()

panel_state_month = panel_state_month[
    ~panel_state_month["State"].isin(states_with_vrm_gaps)
].copy()

State-month rows with missing VRM: 24
States with missing VRM months (top):
State
Vermont    24
dtype: int64


## 9) Export clean CSV (ready to merge with external predictors)

Merge keys you will use later:
- `State`, `Year`, `Month`

This will allow clean merges with:
- BLS unemployment (state-month)
- policy stringency/mobility (state-month after aggregation)
- fuel prices (region-to-state mapping if needed)
- red/blue (state-level)
- workforce composition (state-year)

In [93]:
#panel_output_path = CLEAN_DIR / OUT_PANEL
panel_state_month = panel_state_month.rename(columns={
    "State": "state",
    "Year": "year",
    "Month": "month"
})
panel_output_path = DATA_CLEAN / OUT_PANEL

panel_state_month.to_csv(panel_output_path, index=False)

print("Saved to:", panel_output_path)

Saved to: /Users/emerino/Documents/Capstone_Project_Group_4-/Data/Cleaned/ridership_state_month_panel_MB_complete_fta_dec2025.csv


## 10) Load panel into SQLite database

In [95]:
# --- Connect to SAME SQLite file ---
db_path = PROJECT_ROOT / "Capstone.db"
con = sqlite3.connect(db_path)
%load_ext sql
%sql sqlite:///{db_path}

# --- Read cleaned CSV ---
df = pd.read_csv(DATA_CLEAN / "ridership_state_month_panel_MB_complete_fta_dec2025.csv")


print("States after filter:", df["state"].nunique())  # should be 48

# --- Write to SQLite ---
df.to_sql(
    "Ridership",
    con,
    if_exists="replace",
    index=False,
    method="multi"
)

print("Ridership table overwritten")

The sql extension is already loaded. To reload it, use:
  %reload_ext sql
States after filter: 48
Ridership table overwritten


In [96]:
%sql SELECT * FROM Ridership LIMIT 5;

 * sqlite:////Users/emerino/Documents/Capstone_Project_Group_4-/Capstone.db
Done.


state,year,month,UPT,PostCOVID,VRM
Alabama,2002,1,236779.0,0,189090.0
Alabama,2002,2,218418.0,0,171900.0
Alabama,2002,3,224473.0,0,180495.0
Alabama,2002,4,238948.0,0,189090.0
Alabama,2002,5,253356.0,0,197685.0


In [97]:
%%sql SELECT COUNT(DISTINCT state) AS num_states
FROM Ridership;

 * sqlite:////Users/emerino/Documents/Capstone_Project_Group_4-/Capstone.db
Done.


num_states
48
